In [ ]:
import os
import pandas as pd
import json
import time
import re
import numpy as np
from kafka import KafkaProducer, KafkaAdminClient, TopicPartition
from kafka.admin import NewTopic

# Parameter Koneksi Aiven Kafka Cloud
KAFKA_BROKER = "kafka-90a3cd4-cejors-676945.g.aivencloud.com:28174"
TOPIC_NAME = "unified_jobs"

# Path Sertifikat SSL (pastikan folder ssl/ ada di direktori kerja Anda)
SSL_FOLDER = "ssl"
CA_FILE = os.path.join(SSL_FOLDER, "ca.pem")
CERT_FILE = os.path.join(SSL_FOLDER, "service.cert")
KEY_FILE = os.path.join(SSL_FOLDER, "service.key")

# Verifikasi file sertifikat
for f in [CA_FILE, CERT_FILE, KEY_FILE]:
    if os.path.exists(f):
        print(f"  [OK] {f}")
    else:
        print(f"  [GAGAL] {f} TIDAK DITEMUKAN!")

print("\nLibrary dan parameter koneksi siap.")

  [OK] ssl/ca.pem
  [OK] ssl/service.cert
  [OK] ssl/service.key

Library dan parameter koneksi siap.


In [25]:
try:
  admin_client = KafkaAdminClient(
      bootstrap_servers=KAFKA_BROKER,
      security_protocol='SSL',
      ssl_cafile=CA_FILE,
      ssl_certfile=CERT_FILE,
      ssl_keyfile=KEY_FILE
  )

  existing_topics = admin_client.list_topics()
  print(f"koneksi ke Aiven kafka berhasil")
  print(f"Topik yang aktif saatini: {existing_topics}")
  admin_client.close()

except Exception as e:
  print(f"Gagal terhubung ke Aiven Kafka: {e}")
  print("pastikan service Aiven Anda sedang running dan sertifikat SSL sudah benar.")

koneksi ke Aiven kafka berhasil
Topik yang aktif saatini: ['unified_jobs', '__consumer_offsets']


In [26]:
def delete_old_topic():
  admin_client = KafkaAdminClient(
      bootstrap_servers=KAFKA_BROKER,
      security_protocol="SSL",
      ssl_cafile=CA_FILE,
      ssl_certfile=CERT_FILE,
      ssl_keyfile=KEY_FILE
  )

  existing_topics = admin_client.list_topics()
  print(f"Topik yang ada saat ini: {existing_topics}")

  if TOPIC_NAME in existing_topics:
    print(f"Menghapus topik '{TOPIC_NAME}'...")
    admin_client.delete_topics([TOPIC_NAME])
    print(f"Topik '{TOPIC_NAME}' berhasil dihapus.")
    print("Menunggu 5 detik agar broker menyelesaikan penghapusan...")
    time.sleep(5)
  else:
    print(f"Topik '{TOPIC_NAME}' tidak ditemukan, tidak perlu dihapus.")

  admin_client.close()

delete_old_topic()


Topik yang ada saat ini: ['unified_jobs', '__consumer_offsets']
Menghapus topik 'unified_jobs'...
Topik 'unified_jobs' berhasil dihapus.
Menunggu 5 detik agar broker menyelesaikan penghapusan...


In [27]:
def create_fresh_topic():
  admin_client = KafkaAdminClient(
      bootstrap_servers=KAFKA_BROKER,
      security_protocol="SSL",
      ssl_cafile=CA_FILE,
      ssl_certfile=CERT_FILE,
      ssl_keyfile=KEY_FILE
  )

  existing_topics = admin_client.list_topics()

  if TOPIC_NAME not in existing_topics:
    print(f"Membuat topik baru '{TOPIC_NAME}'...")
    topic = NewTopic(
        name=TOPIC_NAME,
        num_partitions=1,
        replication_factor=1
    )
    admin_client.create_topics(new_topics=[topic], validate_only=False)
    print(f"Topik '{TOPIC_NAME}' berhasil dibuat!")
  else:
    print(f"Topik '{TOPIC_NAME}' sudah ada dan siap digunakan.")

  admin_client.close()

create_fresh_topic()


Membuat topik baru 'unified_jobs'...
Topik 'unified_jobs' berhasil dibuat!


In [28]:
print("=" * 60)
print("[Extraction] Membaca Dataset Adzuna Api")
print("=" * 60)

adzuna_csv = "Adzuna API/adzuna_jobs.csv"
df_adz = pd.read_csv(adzuna_csv)

print(f"File  : {adzuna_csv}")
print(f"Baris : {len(df_adz):,}")
print(f"Kolom : {list(df_adz.columns)}")
print(f"Null  :")
print(df_adz.isnull().sum()[df_adz.isnull().sum() > 0])
print()
df_adz.head(3)

[Extraction] Membaca Dataset Adzuna Api
File  : Adzuna API/adzuna_jobs.csv
Baris : 280
Kolom : ['job_id', 'title', 'description', 'company_name', 'location', 'location_area', 'min_salary', 'max_salary', 'salary_is_predicted', 'contract_time', 'contract_type', 'created_time', 'category_tag', 'category_label', 'search_role', 'redirect_url']
Null  :
contract_time    218
contract_type    245
dtype: int64



,job_id,title,description,company_name,location,location_area,min_salary,max_salary,salary_is_predicted,contract_time,contract_type,created_time,category_tag,category_label,search_role,redirect_url
0,5712067843,Data Engineer,ManTech seeks a Data Engineer to support our I...,ManTech International,"Chantilly, Fairfax County","US, Virginia, Fairfax County, Chantilly",115000.00,160000.00,0,NaN,NaN,2026-04-27T19:15:00Z,it-jobs,IT Jobs,Data Engineer,https://www.adzuna.com/land/ad/5712067843?se=a...
1,5712067907,Senior Data Engineer,ManTech International seeks a Senior Data Engi...,ManTech International,"Chantilly, Fairfax County","US, Virginia, Fairfax County, Chantilly",145000.00,180000.00,0,NaN,NaN,2026-04-27T19:15:01Z,it-jobs,IT Jobs,Data Engineer,https://www.adzuna.com/land/ad/5712067907?se=a...
2,5728506923,"Principal, Data Engineering",Cargill is committed to providing food and agr...,Cargill,"Atlanta, Fulton County","US, Georgia, Fulton County, Atlanta",145712.25,145712.25,1,NaN,NaN,2026-05-13T17:43:28Z,it-jobs,IT Jobs,Data Engineer,https://www.adzuna.com/land/ad/5728506923?se=a...


In [29]:
print("=" * 60)
print("[EXTRACTION] Membaca Dataset Kaggle LinkedIn (TANPA LIMIT)")
print("=" * 60)

kaggle_csv = "kaggle - scaraping job from linkedin 2023 - 2024/postings.csv"
df_kag = pd.read_csv(kaggle_csv)

print(f"File  : {kaggle_csv}")
print(f"baris : {len(df_kag):,}")
print(f"Kolom : {len(df_kag.columns)} kolom")
print(f"\nStatistik Null (kolom yang akan kita gunakan): ")
col_interest = ['job_id', 'title', 'description', 'company_name', 'location', 'min_salary', 'max_salary', 'work_type', 'formatted_experience_level', 'skills_desc']
for col in col_interest:
  null_count = df_kag[col].isnull().sum()
  pct = (null_count / len(df_kag)) * 100
  print(f" {col:35s} -> {null_count:>6,} null ({pct:.1f}%)")

print(f"\nDuplikat job_id   : {df_kag['job_id'].duplicated().sum():,}")
print(f"Duplikat title+desc : {df_kag.duplicated(subset=['title','description']).sum():,}")
print()
df_kag.head(3)


[EXTRACTION] Membaca Dataset Kaggle LinkedIn (TANPA LIMIT)
File  : kaggle - scaraping job from linkedin 2023 - 2024/postings.csv
baris : 123,849
Kolom : 31 kolom

Statistik Null (kolom yang akan kita gunakan): 
 job_id                              ->      0 null (0.0%)
 title                               ->      0 null (0.0%)
 description                         ->      7 null (0.0%)
 company_name                        ->  1,719 null (1.4%)
 location                            ->      0 null (0.0%)
 min_salary                          -> 94,056 null (75.9%)
 max_salary                          -> 94,056 null (75.9%)
 work_type                           ->      0 null (0.0%)
 formatted_experience_level          -> 29,409 null (23.7%)
 skills_desc                         -> 121,410 null (98.0%)

Duplikat job_id   : 0
Duplikat title+desc : 12,944



,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0


In [30]:
print("="*60)
print("[EXTRACTION] Membaca dataset O*net")
print("="*60)

occ_path = "ONET Occupational Database/db_30_2_excel/Occupation Data.xlsx"
tech_path = "ONET Occupational Database/db_30_2_excel/Technology Skills.xlsx"

df_occ = pd.read_excel(occ_path)
df_tech = pd.read_excel(tech_path)

print(f"\nOccupation Data:")
print(f"  Baris  : {len(df_occ):,} profesi")
print(f"  Kolom  : {list(df_occ.columns)}")
print(f"\nTechnology Skills:")
print(f"  Baris  : {len(df_tech):,} entri skill")
print(f"  Kolom  : {list(df_tech.columns)}")

# Preview pengelompokan skill per profesi
tech_grouped = df_tech.groupby('O*NET-SOC Code')['Example'].apply(
    lambda x: ', '.join(x.astype(str).unique())
).reset_index()
tech_grouped.columns = ['O*NET-SOC Code', 'skills_aggregated']

print(f"\nSetelah agregasi, setiap profesi memiliki daftar skill gabungan.")
print(f"Contoh:")
print(tech_grouped.head(2).to_string(index=False))

[EXTRACTION] Membaca dataset O*net

Occupation Data:
  Baris  : 1,016 profesi
  Kolom  : ['O*NET-SOC Code', 'Title', 'Description']

Technology Skills:
  Baris  : 32,773 entri skill
  Kolom  : ['O*NET-SOC Code', 'Title', 'Example', 'Commodity Code', 'Commodity Title', 'Hot Technology', 'In Demand']

Setelah agregasi, setiap profesi memiliki daftar skill gabungan.
Contoh:
O*NET-SOC Code                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [31]:
print("=" * 60)
print("RINGKASAN EXTRACTION")
print("=" * 60)
print(f"  Adzuna API         : {len(df_adz):>10,} baris")
print(f"  Kaggle LinkedIn    : {len(df_kag):>10,} baris")
print(f"  O*NET Occupations  : {len(df_occ):>10,} profesi")
print(f"  O*NET Tech Skills  : {len(df_tech):>10,} entri skill")
print(f"  {'':->40}")
total_raw = len(df_adz) + len(df_kag) + len(df_occ)
print(f"  TOTAL DATA MENTAH  : {total_raw:>10,} baris")
print(f"\nData mentah siap untuk masuk ke tahap Transformation & Cleaning.")

RINGKASAN EXTRACTION
  Adzuna API         :        280 baris
  Kaggle LinkedIn    :    123,849 baris
  O*NET Occupations  :      1,016 profesi
  O*NET Tech Skills  :     32,773 entri skill
  ----------------------------------------
  TOTAL DATA MENTAH  :    125,145 baris

Data mentah siap untuk masuk ke tahap Transformation & Cleaning.


In [32]:
print("[TRANSFORM] Memetakan data adzuna ke format standar...")

records_adzuna = []

for idx, row in df_adz.iterrows():
    records_adzuna.append({
        'unique_id': f"ADZ_{row['job_id']}",
        'data_source': 'Adzuna_API',
        'job_title': str(row['title']).strip(),
        'company_name': str(row['company_name']).strip() if pd.notna(row['company_name']) else None,
        'job_description': str(row['description'])[:2000].strip() if pd.notna(row['description']) else None,
        'location': str(row['location']).strip() if pd.notna(row['location']) else None,
        'min_salary': float(row['min_salary']) if pd.notna(row['min_salary']) else None,
        'max_salary': float(row['max_salary']) if pd.notna(row['max_salary']) else None,
        'contract_time': str(row['contract_time']).strip() if pd.notna(row['contract_time']) else None,
        'contract_type': str(row['contract_type']).strip() if pd.notna(row['contract_type']) else None,
        'experience_level': 'Not Specified',
        'skills_required': None
    })

print(f"  Adzuna berhasil dipetakan: {len(records_adzuna):,} baris")

[TRANSFORM] Memetakan data adzuna ke format standar...
  Adzuna berhasil dipetakan: 280 baris


In [33]:
print("[TRANSFORM] Memetakan data Kaggle LinkedIn ke format standar...")
print("  (Proses ini memakan waktu beberapa detik karena dataset besar)")

records_kaggle = []

for idx, row in df_kag.iterrows():
    # Lewati baris yang tidak memiliki judul atau deskripsi
    if pd.isna(row['title']) or pd.isna(row['description']):
        continue

    records_kaggle.append({
        'unique_id': f"KAG_{row['job_id']}",
        'data_source': 'Kaggle_LinkedIn',
        'job_title': str(row['title']).strip(),
        'company_name': str(row['company_name']).strip() if pd.notna(row['company_name']) else None,
        'job_description': str(row['description'])[:2000].strip() if pd.notna(row['description']) else None,
        'location': str(row['location']).strip() if pd.notna(row['location']) else None,
        'min_salary': float(row['min_salary']) if pd.notna(row['min_salary']) else None,
        'max_salary': float(row['max_salary']) if pd.notna(row['max_salary']) else None,
        'contract_time': str(row['work_type']).strip() if pd.notna(row['work_type']) else None,
        'contract_type': 'Not Specified',
        'experience_level': str(row['formatted_experience_level']).strip() if pd.notna(row['formatted_experience_level']) else 'Not Specified',
        'skills_required': str(row['skills_desc'])[:1000].strip() if pd.notna(row['skills_desc']) else None
    })

    if (idx + 1) % 25000 == 0:
        print(f"  ... {idx + 1:,} baris diproses")

print(f"  Kaggle berhasil dipetakan: {len(records_kaggle):,} baris")

[TRANSFORM] Memetakan data Kaggle LinkedIn ke format standar...
  (Proses ini memakan waktu beberapa detik karena dataset besar)
  ... 25,000 baris diproses
  ... 50,000 baris diproses
  ... 75,000 baris diproses
  ... 100,000 baris diproses
  Kaggle berhasil dipetakan: 123,842 baris


In [34]:
print("[TRANSFORM] Memetakan data O*net ke format standar..")

# Gabungkan occupation data dengan techcnology skills yang sudah di agregasi
df_onet_merged = pd.merge(df_occ, tech_grouped, on='O*NET-SOC Code',how='left')
records_onet = []
for idx, row in df_onet_merged.iterrows():
  records_onet.append({
      'unique_id': f"ONE_{row['O*NET-SOC Code']}",
      'data_source': 'ONET_Standard',
      'job_title': str(row['Title']).strip(),
      'company_name': 'ONET Standard System',
      'job_description': str(row['Description'])[:2000].strip() if pd.notna(row['Description']) else None,
      'location': 'Global',
      'min_salary': None,
      'max_salary': None,
      'contract_time': 'Standard',
      'contract_type': 'Standard',
      'experience_level': 'Standard',
      'skills_required': str(row['skills_aggregated'])[:1000] if pd.notna(row['skills_aggregated']) else None
  })

print(f"O*NET berhasil dipetakan: {len(records_onet):,} baris")

[TRANSFORM] Memetakan data O*net ke format standar..
O*NET berhasil dipetakan: 1,016 baris


In [35]:
# Cell 12 : Menggabungkan seluruh data

print("[TRANSFORM] Menggabungkan seluruh data ...")

all_records = records_adzuna + records_kaggle + records_onet

df_unified = pd.DataFrame(all_records)

print(f"Total data gabungan (sebelum cleaning): {len(df_unified):,} baris")
print(f"\n Komposisi per sumber data:")
print(df_unified['data_source'].value_counts().to_string())

[TRANSFORM] Menggabungkan seluruh data ...


Total data gabungan (sebelum cleaning): 125,138 baris

 Komposisi per sumber data:
data_source
Kaggle_LinkedIn    123842
ONET_Standard        1016
Adzuna_API            280


In [36]:
# cell 13 : Data Cleaning Menyeluruh

print("="*60)
print("[CLEANING] Memulai proses pembersihan data menyeluruh")
print("="*60)

df_clean = df_unified.copy()
print(f"\n [0] ukuran awal : {len(df_clean):>10,} baris")

# cleaning 1: hapus baris tanpa deskripsi pekerjaan ---
df_clean = df_clean.dropna(subset=['job_title', 'job_description'])
print(f" [1] setelah hapus null title/description : {len(df_clean):>10,} baris")

# cleaning 2: hapus duplikat berdasarkan unique_id ---
df_clean = df_clean.drop_duplicates(subset=['unique_id'])
print(f" [2] setelah hapus duplikat unique_id : {len(df_clean):>10,} baris")

# cleaning 3: hapus spam recruiter (title + description kembar) ---
df_clean = df_clean.drop_duplicates(subset=['job_title', 'job_description'])
print(f" [3] setelah hapus spam rekruter : {len(df_clean):>10,} baris")

# cleaning 4: seragamkan casing pada contract_time ---
df_clean['contract_time'] = df_clean['contract_time'].str.upper()
unique_ct = df_clean['contract_time'].dropna().unique()
print(f" [4] contract_time dseragamkan : {len(df_clean):>10,} baris")

# cleaning 5: Harmonisasi lokasi yang sinonim ---
loc_mapping = {
    "US": "United States",
    "New York City Metropolitan Area": "New York, NY",
    "Austin, Texas Metropolitan Area": "Austin, TX",
    "San Fransisco Bay Area": "San Francisco, CA",
    "Greater Chicago Area": "Chicago, IL",
    "Greater Boston": "Boston, MA",
    "Dallas-Fort Worth Metroplex": "Dallas, TX",
    "Greater Seattle Area": "Seattle, WA",
    "Greater Los Angeles Area": "Los Angeles, CA",
    "Greater Philadelphia": "Philadelphia, PA",
    "Greater Denver Area": "Denver, CO",
    "Greeater Minneapolis-St. Paul Area": "Minneapolis, MN",
    "Washington DC-Baltimore Area": "Washington, DC",
    "Greater Tampa Bay Area": "Tampa, FL",
    "Greater Portland Oregon Area": "Portland, OR",
    "Greater Scramento Area": "Sacramento, CA",
    "Greater Nashville Area, TN": "Nashville, TN",
}
df_clean['location'] = df_clean['location'].replace(loc_mapping)
print(f" [5] Lokasi sinonim diharmonisasi : {len(loc_mapping)} Pemetaan diterapkan")

# cleaning 6: menghapus whitespace berlebih pada semua kolom string ---
str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols:
  df_clean[col] = df_clean[col].str.strip()
print(f" [6] whitespace berlebihan dibersihkan : {len(str_cols)} kolom string")

# cleaning 7: Ganti NaN dengan None untuk kompatibilitas JSON Kafka ---
df_clean = df_clean.replace({np.nan: None})
print(f" [7] NaN dikonversi ke None (JSON-safe)")

# Hasil AKhir
print(f"\n{'=' * 60}")
print(f"Data bersih akhir: {len(df_clean):,} baris siap dikirim ke aiven")
print(f"{'=' * 60}")
print(f"\n Komposisi per sumber data:")
print(df_clean['data_source'].value_counts().to_string())

[CLEANING] Memulai proses pembersihan data menyeluruh

 [0] ukuran awal :    125,138 baris
 [1] setelah hapus null title/description :    125,138 baris
 [2] setelah hapus duplikat unique_id :    125,138 baris
 [3] setelah hapus spam rekruter :    108,963 baris
 [4] contract_time dseragamkan :    108,963 baris
 [5] Lokasi sinonim diharmonisasi : 17 Pemetaan diterapkan
 [6] whitespace berlebihan dibersihkan : 10 kolom string
 [7] NaN dikonversi ke None (JSON-safe)

Data bersih akhir: 108,963 baris siap dikirim ke aiven

 Komposisi per sumber data:
data_source
Kaggle_LinkedIn    107788
ONET_Standard        1016
Adzuna_API            159


/tmp/ipykernel_25463/2864179758.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df_clean.select_dtypes(include='object').columns


In [37]:
# cell 14: Inspeksi kualitas data bersih

print("Laporan kualitas data bersih")

# cek duplikat
print(f"\n Sisa duplikat unique_id : {df_clean.duplicated(subset=['unique_id']).sum()}")
print(f" Sisa duplikat title+desc : {df_clean.duplicated(subset=['job_title', 'job_description']).sum()}")

# Cek null per kolom
print(f"\n Statistik null per kolom:")
for col in df_clean.columns:
  null_count = df_clean[col].isnull().sum()
  pct = (null_count / len(df_clean)) * 100
  status = "Bersih" if null_count == 0 else f"{null_count:,} null ({pct:.1f}%)"
  print(f" {col:25s} -> {status}")

# cek casing contract_time
print(f"\n top 10 lokasi:")
print(f" {df_clean['location'].value_counts().head(10).to_string()}")
print(f"\nData siap untuk distream ke aiven kafka")

Laporan kualitas data bersih

 Sisa duplikat unique_id : 0
 Sisa duplikat title+desc : 0

 Statistik null per kolom:
 unique_id                 -> Bersih
 data_source               -> Bersih
 job_title                 -> Bersih
 company_name              -> 1,698 null (1.6%)
 job_description           -> Bersih
 location                  -> Bersih
 min_salary                -> 81,027 null (74.4%)
 max_salary                -> 81,027 null (74.4%)
 contract_time             -> 156 null (0.1%)
 contract_type             -> 157 null (0.1%)
 experience_level          -> Bersih
 skills_required           -> 105,966 null (97.2%)

 top 10 lokasi:
 location
United States     7761
New York, NY      3358
Chicago, IL       1855
Houston, TX       1556
Dallas, TX        1471
Atlanta, GA       1244
Austin, TX        1210
Boston, MA        1158
Washington, DC    1057
Global            1016

Data siap untuk distream ke aiven kafka


In [38]:
# cell 15: inisialisasi kafka producer ssl
print("[Kafka] menginisialisasi producer dengan enkripsi ssl...")

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    security_protocol="SSL",
    ssl_cafile=CA_FILE,
    ssl_certfile=CERT_FILE,
    ssl_keyfile=KEY_FILE,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    request_timeout_ms=300000, # timeout 5 menit
    max_block_ms=300000,
    retries=10,
    linger_ms=100, # batch selama 100 ms sebellum kirim
    batch_size=32768 # batch size 32kb
)

print("producer berhasil terhubung dengan aiven kafka cloud")
print(f"target: {len(df_clean):,} pesan akan dikirim ke topik '{TOPIC_NAME}'")

[Kafka] menginisialisasi producer dengan enkripsi ssl...
producer berhasil terhubung dengan aiven kafka cloud
target: 108,963 pesan akan dikirim ke topik 'unified_jobs'


In [39]:
# CELL 16: Streaming seluruh data bersih ke aiven kafka
print(f"Memulai streaming {len(df_clean):,} data bersih ke aiven kafka")

records_to_send = df_clean.to_dict(orient='records')

success_count = 0
fail_count = 0
start_time = time.time()

for idx, record in enumerate(records_to_send):
  try:
    future = producer.send(TOPIC_NAME, value=record)
    future.get(timeout=10)
    success_count += 1

    # Jeda pengaman anti-limit (15 milidetik per pesan)
    time.sleep(0.01)

    # Cetak progres setiap 1.000 pesan
    if success_count % 1000 == 0:
      elapsed = time.time() - start_time
      speed = success_count / elapsed
      remaining = (len(records_to_send) - success_count) / speed
      print(f" [{success_count:>7,} / {len(records_to_send):,}] "
            f"Waktu: {elapsed:.0f}s | "
            f"Kecepatan: {speed:.0f} msg/s | "
            f"Estimasi sisa: {remaining:.0f}s"
            )

  except Exception as e:
    fail_count += 1
    if fail_count <=5:
      print(f"Gagal mengirim {record.get('unique_id')}: {e}")
    elif fail_count == 6:
      print(f" ...pesan error berikutnya tidak ditampilkan...")

# Pastikan semua pesan yang masih di buffer terkirim sepenuhnya
print("\n[Kafka] melakukan flushing buffer akhir...")
producer.flush()

duration = time.time() - start_time

print('streaming selesai')
print(f"berhasil terkirim : {success_count:,} pesan")
print(f"gagal terkirim    : {fail_count:,} pesan")
print(f"waktu total       : {duration:.1f} detik ({duration/60:.1f} menit)")


Memulai streaming 108,963 data bersih ke aiven kafka
 [  1,000 / 108,963] Waktu: 295s | Kecepatan: 3 msg/s | Estimasi sisa: 31864s
 [  2,000 / 108,963] Waktu: 588s | Kecepatan: 3 msg/s | Estimasi sisa: 31432s
Gagal mengirim KAG_3884807274: KafkaTimeoutError: Timeout after waiting for 10 secs.
Gagal mengirim KAG_3884807303: KafkaTimeoutError: Timeout after waiting for 10 secs.
 [  3,000 / 108,963] Waktu: 921s | Kecepatan: 3 msg/s | Estimasi sisa: 32534s
 [  4,000 / 108,963] Waktu: 1210s | Kecepatan: 3 msg/s | Estimasi sisa: 31742s
 [  5,000 / 108,963] Waktu: 1507s | Kecepatan: 3 msg/s | Estimasi sisa: 31331s
 [  6,000 / 108,963] Waktu: 1827s | Kecepatan: 3 msg/s | Estimasi sisa: 31344s
 [  7,000 / 108,963] Waktu: 2120s | Kecepatan: 3 msg/s | Estimasi sisa: 30887s
 [  8,000 / 108,963] Waktu: 2400s | Kecepatan: 3 msg/s | Estimasi sisa: 30284s
 [  9,000 / 108,963] Waktu: 2689s | Kecepatan: 3 msg/s | Estimasi sisa: 29863s
 [ 10,000 / 108,963] Waktu: 2966s | Kecepatan: 3 msg/s | Estimasi sis

In [40]:
# cell 17: menutup koneksi producer

producer.close()
print("koneksi kafka produser ditutup dengan aman.")

koneksi kafka produser ditutup dengan aman.


In [41]:
from kafka import KafkaConsumer, TopicPartition
# cell 18: verifikasi jumlah data di aiven kkafka

print("[VERIFIKASI] memeriksa data di broker aiven kafka...")

consumer = KafkaConsumer(
    bootstrap_servers=KAFKA_BROKER,
    security_protocol="SSL",
    ssl_cafile=CA_FILE,
    ssl_certfile=CERT_FILE,
    ssl_keyfile=KEY_FILE
)

# Ambil informasi partisi untuk topik unified_jobs
partitions = consumer.partitions_for_topic(TOPIC_NAME)
if partitions:
  tps = [TopicPartition(TOPIC_NAME, p) for p in partitions]

  # Dapatkan offset awal dan akhir
  beginning = consumer.beginning_offsets(tps)
  ending = consumer.end_offsets(tps)

  total_messages = 0
  print(f"\n Topik: {TOPIC_NAME}")
  print(f" jumlah Partisi: {len(partitions)}")
  print(f" {'Partisi':>10} | {'Offset awal':>12} | {'Offset akhir':>12} | {'Jumlah pesan':>12}")
  print(f" {'-'*10} | {'-'*12} | {'-'*12} | {'-'*12}")

  for tp in tps:
    count = ending[tp] - beginning[tp]
    total_messages += count
    print(f" {tp.partition:>10} | {beginning[tp]:>12,} | {ending[tp]:>12,} | {count:>12,}")

  print(f"\n Total pesan di aiven: {total_messages:,}")
  print(f" Target yang dikirim: {len(df_clean):,}")

  if total_messages == len(df_clean):
    print(f"\n Verifikasi berhasil: Jumlah data cocok 100 %")
  else:
    print(f"perhatian: ada selisih {abs(total_messages - len(df_clean)):,} pesan.")
    print(f" Hal ini bisa terjadi jika streaming sebelumnya terputus ditengah.")
else:
  print(f" Topik '{TOPIC_NAME}' tidak ditemukan!")

consumer.close()

[VERIFIKASI] memeriksa data di broker aiven kafka...



 Topik: unified_jobs
 jumlah Partisi: 1
    Partisi |  Offset awal | Offset akhir | Jumlah pesan
 ---------- | ------------ | ------------ | ------------
          0 |            0 |      108,964 |      108,964

 Total pesan di aiven: 108,964
 Target yang dikirim: 108,963
perhatian: ada selisih 1 pesan.
 Hal ini bisa terjadi jika streaming sebelumnya terputus ditengah.


In [42]:
# cell 19: Menarik data dari aiven cache lokal streamlit

import uuid

print("[CACHE] Menarik seluruh data bersih dari aiven kafka ke file cache lokal...")
random_group_id = f"cache-refresh-{uuid.uuid4().hex[:8]}"
consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers=KAFKA_BROKER,
    security_protocol="SSL",
    ssl_cafile=CA_FILE,
    ssl_certfile=CERT_FILE,
    ssl_keyfile=KEY_FILE,
    auto_offset_reset="earliest",
    group_id=random_group_id,
    enable_auto_commit=False,
    value_deserializer=lambda v: json.loads(v.decode('utf-8'))
)

# Tunggu pastisi di assign

print(" Menunggu alokasi partisi...")
assigned = []
start_wait = time.time()
while not assigned:
  consumer.poll(timeout_ms=1000)
  assigned = list(consumer.assignment())
  if time.time() - start_wait > 20:
    break

if not assigned:
  print(" Gagal: Tidak mendapat alokasi partisi.")
else:
  print(f" Terhubung ke {len(assigned)} partisi.")

  # Dapatkan end offsets untuk menentukan target
  end_offsets = consumer.end_offsets(assigned)

  records = []
  poll_start = time.time()

  while True:
    # Cek apakah semua partisi sudah dibaca sampai akhir
    all_done = all(consumer.position(tp) >= end_offsets[tp] for tp in assigned)
    if all_done:
      break

    msg_pack = consumer.poll(timeout_ms=3000)
    if msg_pack:
      for tp, messages in msg_pack.items():
        for msg in messages:
          records.append(msg.value)

      if len(records) % 5000 == 0:
        print(f" ... {len(records):,} pesan diterima")

    # Tiemout pengaman 5 menit
    if time.time() - poll_start > 300:
      print(" Timeout tercapai.")
      break

  consumer.close()

  if records:
    df_cache = pd.DataFrame(records)

    # SImpan ke folder pnm_dashboard_app

    cache_path = "pnm_dashboard_app/cached_data.csv"
    df_cache.to_csv(cache_path, index=False)

    print(f"\n Cache berhasil diperbarui")
    print(f" File   : {cache_path}")
    print(f" Baris  : {len(df_cache):,}")
    print(f" ukuran : { os.path.getsize(cache_path) / (1024*1024):.1f} MB")
  else:
    print(" gagal: tidak ada data yang diterima dari broker.")

[CACHE] Menarik seluruh data bersih dari aiven kafka ke file cache lokal...
 Menunggu alokasi partisi...
 Terhubung ke 1 partisi.
 Timeout tercapai.

 Cache berhasil diperbarui
 File   : pnm_dashboard_app/cached_data.csv
 Baris  : 10,210
 ukuran : 18.9 MB


In [43]:
# CELL 20: Ringkasan Akhir Pipeline ETL

print("PIPELINE ETL SELESAI DENGAN SUKSES!")

print(f"""
  Rangkuman Proses:
  1. Data mentah diekstrak dari 3 sumber:
     - Adzuna API       : {len(df_adz):>10,} baris
     - Kaggle LinkedIn  : {len(df_kag):>10,} baris
     - O*NET Standard   : {len(df_occ):>10,} profesi

  2. Data dibersihkan secara menyeluruh:
     - Null title/description dihapus
     - Duplikat ID dihapus
     - Spam recruiter dihapus
     - Casing diseragamkan
     - Lokasi diharmonisasi
     - Whitespace dibersihkan

  3. Data bersih dikirim ke Aiven Kafka:
     - Topik             : {TOPIC_NAME}
     - Total terkirim    : {success_count:,} baris

  4. Cache lokal Streamlit diperbarui:
     - File              : pnm_dashboard_app/cached_data.csv

  Langkah selanjutnya:
  - Jalankan dashboard Streamlit untuk melihat data bersih
  - Gunakan data bersih untuk pelatihan model Machine Learning
""")

PIPELINE ETL SELESAI DENGAN SUKSES!

  Rangkuman Proses:
  1. Data mentah diekstrak dari 3 sumber:
     - Adzuna API       :        280 baris
     - Kaggle LinkedIn  :    123,849 baris
     - O*NET Standard   :      1,016 profesi

  2. Data dibersihkan secara menyeluruh:
     - Null title/description dihapus
     - Duplikat ID dihapus
     - Spam recruiter dihapus
     - Casing diseragamkan
     - Lokasi diharmonisasi
     - Whitespace dibersihkan

  3. Data bersih dikirim ke Aiven Kafka:
     - Topik             : unified_jobs
     - Total terkirim    : 108,961 baris

  4. Cache lokal Streamlit diperbarui:
     - File              : pnm_dashboard_app/cached_data.csv

  Langkah selanjutnya:
  - Jalankan dashboard Streamlit untuk melihat data bersih
  - Gunakan data bersih untuk pelatihan model Machine Learning

